# Fairness-Aware Citation Ranking: Mitigating Structural Age Bias in Academic Networks
### CSCE 676 :: Data Mining and Analysis :: Texas A&M University :: Spring 2026
**Name:** Brandon Hill | **UIN:** 236004960

## Collaboration & Resources Declaration
On my honor, I declare the following resources:
* **Collaborators:** None.
* **Web Sources:** igraph official documentation (python.igraph.org).
* **AI Tools:** Gemini was prompted to help brainstorm algorithmic feasibility, debug, and format the markdown cells.
* **Citations:**
    * Yushun Dong, Song Wang, Zhenyu Lei, Zaiyi Zheng, Jing Ma, Chen Chen, and Jundong Li. 2025. Fairness-Aware Graph Learning: A Benchmark. In Proceedings of the 31st ACM SIGKDD Conference on Knowledge Discovery and Data Mining V.2 (KDD '25). Association for Computing Machinery, New York, NY, USA, 5402–5412. https://doi.org/10.1145/3711896.3737392
    * Jie Tang, Jing Zhang, Limin Yao, Juanzi Li, Li Zhang, and Zhong Su. ArnetMiner: Extraction and Mining of Academic Social Networks. In Proceedings of the Fourteenth ACM SIGKDD International Conference on Knowledge Discovery and Data Mining (SIGKDD'2008). pp.990-998.
    * Jinyu Yang, Liangwei Yang, Zeyuan Guo, Jiayi Gao, Jing Wu, Tianhao Chai, Hai Huang, Cheng Yang, and Chuan Shi. 2025. Benchmarking Graph Foundation Models. In Proceedings of the 31st ACM SIGKDD Conference on Knowledge Discovery and Data Mining V.2 (KDD '25). Association for Computing Machinery, New York, NY, USA, 5866–5875. https://doi.org/10.1145/3711896.3737410
    * Junhong Lin, Xiaojie Guo, Shuaicheng Zhang, Yada Zhu, and Julian Shun. 2025. When Heterophily Meets Heterogeneity: Challenges and a New Large-Scale Graph Benchmark. In Proceedings of the 31st ACM SIGKDD Conference on Knowledge Discovery and Data Mining V.2 (KDD '25). Association for Computing Machinery, New York, NY, USA, 5607–5618. https://doi.org/10.1145/3711896.3737421

---

## 1. Introduction & Motivation

Every year, billions of dollars are poured into research and development, yet the search engines and recommendation systems used to discover breakthroughs often rely on classic graph algorithms that are structurally rigged against new ideas.

In this project, we analyze the **ACM-Citation-network-V12** dataset (6.6 million nodes, 13.1 million edges). Initial exploratory data analysis revealed a severe **"Matthew Effect" (Age Bias)**. Because citation networks are Directed Acyclic Graphs (DAGs) that grow over time, older papers naturally hoard structural connections simply because they have existed longer. Standard global centrality metrics mistake this "age" for "importance," effectively burying modern innovations in isolated structural silos.

**The Goal:** The purpose of this notebook is to cleanly demonstrate this structural bias using course techniques (Centrality and Community Detection), and then implement an external technique—**Fairness-Aware PageRank (via Biased Teleportation)**—to bridge disconnected research silos and restore visibility to modern research without sacrificing semantic relevance.

In [1]:
# Install high-performance graph library
!pip install python-igraph

import os
import urllib.request
import zipfile
import json
import igraph as ig
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Download ACM-Citation-network-V12
url = "https://opendata.aminer.cn/dataset/ACM-Citation-network-V12.zip"
zip_path = "ACM_V12.zip"
extract_path = "ACM_V12_Data"

if not os.path.exists(zip_path):
    print("Downloading dataset (this may take a few minutes)...")
    urllib.request.urlretrieve(url, zip_path)
    print("Download complete.")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
        print(f"Extracted to {extract_path}")

# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(extract_path)
#     print(f"Extracted to {extract_path}")

# Identify the file
json_file = [f for f in os.listdir(extract_path) if f.endswith('.jsonl')][0]
full_json_path = os.path.join(extract_path, json_file)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 77.7 MB/s eta 0:00:00
Download complete.
Extracted to ACM_V12_Data


In [2]:
# Install required packages (Uncomment if running in fresh Colab instance)
# !pip install python-igraph pandas numpy matplotlib scipy

import igraph as ig
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kendalltau
import time
import json

print(f"igraph version: {ig.__version__}")

# ==============================================================================
# DATA LOADING PLACEHOLDER
# NOTE: Ensure your ACM-V12 graph 'g' is loaded into memory here using your
# memory-safe custom streaming logic from Checkpoint 1.
# ==============================================================================

def process_full_acm_v12(filepath):
    """
    Specifically designed for ACM V12:
    - Robustly handles 'venue' as either a string or a dictionary.
    - Handles the JSON Array format (strips [ , ]).
    - Maps Long IDs to strings and deduplicates papers.
    """
    edges = []
    metadata = {}

    print("Starting full file parse of ACM V12...")

    with open(filepath, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f):
            line = line.strip()

            # Skip the start/end of the JSON array or empty lines
            if not line or line == "[" or line == "]":
                continue

            # Remove trailing comma
            if line.endswith(","):
                line = line[:-1]

            try:
                paper = json.loads(line)
                p_id = str(paper.get('id'))
                if not p_id:
                    continue

                if p_id not in metadata:
                    # Handle 'venue' as dict or str
                    venue_data = paper.get('venue')
                    if isinstance(venue_data, dict):
                        venue_name = venue_data.get('raw', 'N/A')
                    else:
                        venue_name = venue_data if venue_data else 'N/A'

                    metadata[p_id] = {
                        'title': paper.get('title', 'N/A'),
                        'venue': venue_name,
                        'year': paper.get('year', 'N/A'),
                        'n_citation': paper.get('n_citation', 0)
                    }

                refs = paper.get('references', [])
                if refs:
                    for ref in refs:
                        edges.append((p_id, str(ref)))

            except (json.JSONDecodeError, TypeError):
                continue

            if line_num % 500000 == 0 and line_num > 0:
                print(f"Processed {line_num:,} lines...")

    # --- Graph Construction ---
    unique_ids = list(metadata.keys())
    id_map = {id_str: i for i, id_str in enumerate(unique_ids)}

    print(f"\nParsing Complete.")
    print(f"Unique Papers Found: {len(unique_ids):,}")

    # Only keep citations where the cited paper exists in our node list
    valid_edges = [(id_map[u], id_map[v]) for u, v in edges if u in id_map and v in id_map]
    print(f"Valid Internal Citations: {len(valid_edges):,}")

    g = ig.Graph(len(unique_ids), valid_edges, directed=True)
    g.vs['id'] = unique_ids
    g.vs['title'] = [metadata[p_id]['title'] for p_id in unique_ids]
    g.vs['year'] = [metadata[p_id]['year'] for p_id in unique_ids]
    g.vs['venue'] = [metadata[p_id]['venue'] for p_id in unique_ids]

    return g

# Run it on full file
g = process_full_acm_v12(full_json_path)


# Example assertion to ensure the graph is loaded before proceeding:
# assert 'g' in locals(), "Please load the graph object 'g' before running downstream cells."
# print(f"Graph loaded successfully: {g.vcount()} nodes, {g.ecount()} edges.")

igraph version: 1.0.0
Starting full file parse of ACM V12...
Processed 500,000 lines...
Processed 1,000,000 lines...
Processed 1,500,000 lines...
Processed 2,000,000 lines...
Processed 2,500,000 lines...
Processed 3,000,000 lines...
Processed 3,500,000 lines...
Processed 4,000,000 lines...
Processed 4,500,000 lines...
Processed 5,000,000 lines...
Processed 5,500,000 lines...
Processed 6,000,000 lines...
Processed 6,500,000 lines...

Parsing Complete.
Unique Papers Found: 6,611,132
Valid Internal Citations: 13,100,550


## 2. RQ1: Global vs. Local "Importance" and the Matthew Effect
**Question:** How does the global vs. local definition of "importance" shift when evaluating isolated graph components versus the Giant Strongly Connected Component (GSCC)?

**Method:** We compute PageRank (Global Importance) and In-Degree (Local Importance) for both the entire network and the GSCC. We evaluate the shift using **Spearman Rank Correlation** and **Top-K Overlap Percentage**. We also group nodes by publication decade to observe how this structural prestige creates a temporal "Age Bias."

In [19]:
from scipy.stats import spearmanr

def analyze_centrality_and_components(g):
    print("Computing Centrality Metrics for RQ1...")
    start_time = time.time()

    # 1. Compute Global Metrics
    if 'pagerank' not in g.vertex_attributes():
        g.vs['pagerank'] = g.pagerank(directed=True)
    if 'indegree' not in g.vertex_attributes():
        g.vs['indegree'] = g.indegree()

    print(f"Global metrics computed in {time.time() - start_time:.2f} seconds.")

    # 2. Extract GSCC and Compute Localized Metrics
    wcc = g.components(mode='weak')
    gscc = wcc.giant()

    gscc_pr = gscc.pagerank(directed=True)
    gscc_ind = gscc.indegree()

    # 3. Evaluation Criteria: Spearman Rank & Top-K Overlap
    top_k = 100
    global_top_pr = set(np.argsort(g.vs['pagerank'])[::-1][:top_k])
    global_top_ind = set(np.argsort(g.vs['indegree'])[::-1][:top_k])
    global_overlap = len(global_top_pr.intersection(global_top_ind)) / top_k

    spearman_global, _ = spearmanr(g.vs['pagerank'], g.vs['indegree'])
    spearman_gscc, _ = spearmanr(gscc_pr, gscc_ind)

    print("\n--- RQ1 Evaluation Criteria: Component Analysis ---")
    print(f"GSCC Size: {gscc.vcount():,} nodes ({(gscc.vcount()/g.vcount())*100:.2f}% of graph)")
    print(f"Global Graph Spearman Correlation (PageRank vs InDegree): {spearman_global:.4f}")
    print(f"GSCC Spearman Correlation (PageRank vs InDegree): {spearman_gscc:.4f}")
    print(f"Global Top-{top_k} Overlap: {global_overlap*100:.2f}%")

    # 4. Temporal Bias (The Matthew Effect)
    valid_nodes = [v for v in g.vs if str(v['year']).isdigit()]
    df = pd.DataFrame({
        'ID': [v['id'] for v in valid_nodes],
        'Year': [int(v['year']) for v in valid_nodes],
        'PageRank': [v['pagerank'] for v in valid_nodes]
    })
    df['Decade'] = (df['Year'] // 10) * 10
    stats = df.groupby('Decade').agg(Total_Papers=('ID', 'count'), Avg_PageRank=('PageRank', 'mean')).reset_index()
    stats = stats[stats['Total_Papers'] > 100]

    print("\n--- Temporal Distribution of Centrality (Matthew Effect) ---")
    print(stats.to_string(index=False))

# Run the analysis
analyze_centrality_and_components(g)

Computing Centrality Metrics for RQ1...
Global metrics computed in 0.00 seconds.

--- RQ1 Evaluation Criteria: Component Analysis ---
GSCC Size: 1,748,376 nodes (26.45% of graph)
Global Graph Spearman Correlation (PageRank vs InDegree): 0.9973
GSCC Spearman Correlation (PageRank vs InDegree): 0.9351
Global Top-100 Overlap: 18.00%

--- Temporal Distribution of Centrality (Matthew Effect) ---
 Decade  Total_Papers  Avg_PageRank
   1890           944  1.737725e-06
   1940           199  9.570568e-08
   1950          2296  2.246352e-06
   1960         12061  1.895031e-06
   1970         43367  1.319759e-06
   1980        120459  7.042784e-07
   1990        390163  3.629751e-07
   2000       1260121  1.750529e-07
   2010       2718766  1.047417e-07
   2020       2062616  8.786690e-08


### RQ1 Findings
The component analysis confirms extreme structural sparsity: only **26.45%** of the graph belongs to the GSCC.

When evaluating the Spearman Rank Correlation between local importance (In-Degree) and global importance (PageRank), we see a near-perfect correlation globally ($0.9973$). However, within the GSCC, the correlation drops slightly ($0.9351$), indicating that the strict definition of importance shifts when evaluating the dense, localized core versus the broader graph. The low Top-100 overlap ($18.00\%$) further highlights that these global and local metrics favor entirely different sets of papers.

Furthermore, combining this with the temporal metadata reveals a severe "Matthew Effect." The average PageRank of a paper published in the 1950s is approximately 25 times higher than a paper published in the 2020s. Standard centrality algorithms treat the graph as static, meaning older foundational papers hoard structural prestige simply by existing longer.

## 3. RQ2: Community Detection & Semantic Purity
**Question:** Do isolated structural communities map strictly to semantic research fields, or do they represent geographical/institutional silos?

**Method:** We use the **Louvain Modularity** algorithm to partition a dense core subgraph into macro-communities. We then manually validate the semantic purity of these clusters by analyzing the distribution of the Venue/Field of Study (FOS) metadata within the largest extracted communities.

In [20]:
from collections import Counter

def analyze_community_semantics(g):
    print("Running Louvain Community Detection on a dense core subgraph...")

    # Extracting a manageable core to ensure memory safety
    top_degree_nodes = np.argsort(g.vs['indegree'])[::-1][:50000]
    core_subgraph = g.subgraph(top_degree_nodes)
    core_undirected = core_subgraph.as_undirected(mode='collapse')

    partition = core_undirected.community_multilevel()
    print(f"Louvain Modularity Score: {partition.modularity:.4f}")
    print(f"Total Communities Detected: {len(partition)}\n")

    # Add community ID to the core subgraph vertices
    core_subgraph.vs['community'] = partition.membership

    # Evaluation Criteria: Manual Validation of Semantic Purity
    print("--- Semantic Validation (Top 3 Communities by Size) ---")
    community_sizes = Counter(partition.membership)
    top_3_comms = [comm[0] for comm in community_sizes.most_common(3)]

    for i, comm_id in enumerate(top_3_comms, 1):
        # Extract venues for nodes in this community
        venues = [v['venue'] for v in core_subgraph.vs if v['community'] == comm_id and v['venue'] != 'N/A' and v['venue'] != '']
        venue_counts = Counter(venues)

        print(f"Community {comm_id} (Size: {community_sizes[comm_id]} nodes)")
        print("  Dominant Venues/FOS:")
        for venue, count in venue_counts.most_common(3):
            print(f"    - {venue[:50]}: {count} papers")
        print()

# Run the analysis
analyze_community_semantics(g)

Running Louvain Community Detection on a dense core subgraph...
Louvain Modularity Score: 0.7646
Total Communities Detected: 617

--- Semantic Validation (Top 3 Communities by Size) ---
Community 9 (Size: 5679 nodes)
  Dominant Venues/FOS:
    - IEEE Transactions on Pattern Analysis and Machine : 717 papers
    - International Journal of Computer Vision: 313 papers
    - IEEE Transactions on Image Processing: 235 papers

Community 1 (Size: 4762 nodes)
  Dominant Venues/FOS:
    - ACM SIGARCH Computer Architecture News: 184 papers
    - IEEE Transactions on Computers: 181 papers
    - ACM SIGPLAN Notices: 156 papers

Community 12 (Size: 3952 nodes)
  Dominant Venues/FOS:
    - SIGIR: 85 papers
    - Information Processing & Management: 75 papers
    - Knowledge Discovery and Data Mining: 74 papers



### RQ2 Findings
Executing the Louvain Modularity algorithm yielded a remarkably high modularity score of **>0.76**, indicating that the graph features highly distinct, dense sub-communities rather than a uniform distribution.

Our manual validation of Field of Study (FOS) / Venue metadata confirms that these clusters map tightly to semantic research fields rather than random silos. For example, specific communities are heavily dominated by distinct sub-disciplines (e.g., Community 9 groups Computer Vision and Image Processing, while Community 12 tightly groups Information Retrieval and Data Mining venues).

**Connecting RQ1 & RQ2 to RQ3:** Because the graph is so heavily siloed into semantic clusters (RQ2), and because standard random walks get trapped by historical prestige (RQ1), a standard recommendation algorithm will fail to bridge these clusters to find new research. This perfectly justifies our implementation of a biased teleportation vector in RQ3 to force the algorithm to jump across these semantic boundaries.

## 4. RQ3: Fairness-Aware Citation Ranking (External Technique)
**Question:** Can a Fairness-Aware PageRank algorithm mitigate structural age bias without significantly degrading the semantic relevance (utility) of the resulting citation recommendations?

**Method:** Inspired by the *FairWalk* algorithm, we implement an attribute-aware demographic bias into the global teleportation vector of the PageRank algorithm. We define our "minority/disadvantaged" demographic as modern papers (published >= 2018). We will evaluate the success of this method using **Demographic Parity** (fairness) and **Kendall-Tau Distance** (utility).

In [21]:
def evaluate_fair_pagerank_sensitivity(g, sample_size=100000, target_year=2018, top_k=100):
    print(f"Executing Fairness-Aware PageRank Sensitivity Analysis on a {sample_size}-node subgraph...")

    # Extract subgraph for computational safety
    sub = g.subgraph(range(sample_size))

    # Define Target Demographic (>= 2018)
    recent_nodes = set()
    for v in sub.vs:
        year_str = str(v['year'])
        if year_str.isdigit() and int(year_str) >= target_year:
            recent_nodes.add(v.index)

    recent_ratio_total = len(recent_nodes) / sample_size
    print(f"Graph Demographics: {recent_ratio_total*100:.2f}% of nodes are >= {target_year}\n")

    # 1. Baseline PageRank
    pr_base = sub.pagerank()
    top_k_base = np.argsort(pr_base)[::-1][:top_k]
    dp_base = sum(1 for i in top_k_base if i in recent_nodes) / top_k

    # 2. Sensitivity Analysis across multiple Bias Weights
    results = []
    bias_multipliers = [1.1, 1.2, 1.3, 1.4, 1.5, 2.0, 3.0, 5.0, 10.0]

    for weight in bias_multipliers:
        # Construct preference vector
        weights = [weight if v.index in recent_nodes else 1.0 for v in sub.vs]

        # Fair PageRank (Biased Teleportation)
        pr_fair = sub.personalized_pagerank(directed=True, damping=0.85, reset=weights)
        top_k_fair = np.argsort(pr_fair)[::-1][:top_k]
        dp_fair = sum(1 for i in top_k_fair if i in recent_nodes) / top_k

        # Utility Evaluation (Kendall-Tau)
        combined_top_nodes = list(set(list(top_k_base) + list(top_k_fair)))
        base_scores_for_kt = [pr_base[i] for i in combined_top_nodes]
        fair_scores_for_kt = [pr_fair[i] for i in combined_top_nodes]

        tau, p_value = kendalltau(base_scores_for_kt, fair_scores_for_kt)

        results.append({
            "Bias Weight": weight,
            "Demographic Parity": f"{dp_fair*100:.2f}%",
            "Kendall-Tau (Utility)": round(tau, 4)
        })

    df_results = pd.DataFrame(results)
    print(f"Baseline Demographic Parity (Top {top_k}): {dp_base*100:.2f}%\n")
    print(df_results.to_string(index=False))

    return df_results

# Run the sensitivity analysis
df_sensitivity = evaluate_fair_pagerank_sensitivity(g)

Executing Fairness-Aware PageRank Sensitivity Analysis on a 100000-node subgraph...
Graph Demographics: 39.89% of nodes are >= 2018

Baseline Demographic Parity (Top 100): 6.00%

 Bias Weight Demographic Parity  Kendall-Tau (Utility)
         1.1             37.00%                 0.8634
         1.2             42.00%                 0.7834
         1.3             43.00%                 0.7715
         1.4             43.00%                 0.7641
         1.5             48.00%                 0.6933
         2.0             63.00%                 0.3312
         3.0             64.00%                 0.2400
         5.0             64.00%                 0.1489
        10.0             67.00%                 0.0357


### 100k Subgraph Findings: The "Goldilocks" Zone
On a manageable subgraph, the algorithm works flawlessly. The baseline PageRank severely under-represented modern papers (only **6.00%** of the top 100 spots, despite making up **39.89%** of the graph).

By testing multiple weights, we identified an optimal threshold at a **Bias Weight of 1.2**. At this weight, the algorithm achieves a Demographic Parity of **42.00%**—perfectly aligning with the actual population distribution—while maintaining a highly stable Kendall-Tau score of **0.7834**. This proves that on a localized scale, a slight global teleportation bias successfully bridges disconnected structural silos without destroying the semantic utility of the ranking.

## 4.1. RQ3 Extension: The Scale-Up Problem
**Question:** Does the successful teleportation bias weight of 1.2 scale to the entire 6.6-million node macro-graph? To test this, we run an extreme-scale sensitivity analysis.

In [17]:
def evaluate_fair_pagerank_sensitivity(g, sample_size=6000000, target_year=2018, top_k=100):
    print(f"Executing Fairness-Aware PageRank Sensitivity Analysis on a {sample_size}-node subgraph...")

    # Extract subgraph for computational safety
    sub = g.subgraph(range(sample_size))

    # Define Target Demographic (>= 2018)
    recent_nodes = set()
    for v in sub.vs:
        year_str = str(v['year'])
        if year_str.isdigit() and int(year_str) >= target_year:
            recent_nodes.add(v.index)

    recent_ratio_total = len(recent_nodes) / sample_size
    print(f"Graph Demographics: {recent_ratio_total*100:.2f}% of nodes are >= {target_year}\n")

    # 1. Baseline PageRank
    pr_base = sub.pagerank()
    top_k_base = np.argsort(pr_base)[::-1][:top_k]
    dp_base = sum(1 for i in top_k_base if i in recent_nodes) / top_k

    # 2. Sensitivity Analysis across multiple Bias Weights
    results = []
    # bias_multipliers = [1.1, 1.2, 1.3, 1.4, 1.5, 2.0, 3.0, 5.0, 10.0, 20.0]
    bias_multipliers = [50.0, 100.0, 500.0, 1000.0, 5000.0]

    for weight in bias_multipliers:
        # Construct preference vector
        weights = [weight if v.index in recent_nodes else 1.0 for v in sub.vs]

        # Fair PageRank (Biased Teleportation)
        pr_fair = sub.personalized_pagerank(directed=True, damping=0.85, reset=weights)
        top_k_fair = np.argsort(pr_fair)[::-1][:top_k]
        dp_fair = sum(1 for i in top_k_fair if i in recent_nodes) / top_k

        # Utility Evaluation (Kendall-Tau)
        combined_top_nodes = list(set(list(top_k_base) + list(top_k_fair)))
        base_scores_for_kt = [pr_base[i] for i in combined_top_nodes]
        fair_scores_for_kt = [pr_fair[i] for i in combined_top_nodes]

        tau, p_value = kendalltau(base_scores_for_kt, fair_scores_for_kt)

        results.append({
            "Bias Weight": weight,
            "Demographic Parity": f"{dp_fair*100:.2f}%",
            "Kendall-Tau (Utility)": round(tau, 4)
        })

    df_results = pd.DataFrame(results)
    print(f"Baseline Demographic Parity (Top {top_k}): {dp_base*100:.2f}%\n")
    print(df_results.to_string(index=False))

    return df_results

# Run the sensitivity analysis
df_sensitivity = evaluate_fair_pagerank_sensitivity(g)

Executing Fairness-Aware PageRank Sensitivity Analysis on a 6000000-node subgraph...
Graph Demographics: 41.60% of nodes are >= 2018

Baseline Demographic Parity (Top 100): 1.00%

 Bias Weight Demographic Parity  Kendall-Tau (Utility)
        50.0              1.00%                 0.2846
       100.0              1.00%                 0.2697
       500.0              1.00%                 0.2522
      1000.0              1.00%                 0.2507
      5000.0              1.00%                 0.2493


### Extreme Scale Sensitivity Findings: The Limits of Global Teleportation
The extreme scale sensitivity analysis revealed a critical limitation in applying global teleportation bias to massive, temporally evolving networks.

While a bias weight of 1.2 successfully achieved demographic parity on a 100,000-node subgraph, applying weights as high as **5000.0** on the 6-million-node graph failed to raise the Demographic Parity above the **1.00%** baseline. Furthermore, the Kendall-Tau utility score degraded to 0.2493, indicating severe ranking distortion without any fairness gain.

This mathematical failure is driven by two graph-theoretic forces:
1. **Probability Dilution:** The 15% teleportation probability mass is distributed across over 2.5 million modern papers. At this scale, even an extreme bias weight results in an infinitesimally small probability boost for any individual node.
2. **Hub Gravity:** Because citation networks are Directed Acyclic Graphs, older foundational papers act as inescapable structural sinks. Even when the teleportation vector forces the random surfer to a modern paper, the subsequent edge-traversal steps (85% probability) immediately pull the surfer backward in time, trapping them in historical hubs with tens of thousands of incoming edges.

In [18]:
def run_fair_pagerank_full_graph(g, optimal_weight=1.2, target_year=2018):
    print("======================================================")
    print(" FINAL EXECUTION: FAIRNESS-AWARE PAGERANK (FULL GRAPH)")
    print("======================================================")
    print(f"Total Nodes: {g.vcount():,}")
    print(f"Total Edges: {g.ecount():,}")
    print(f"Applying Optimal Bias Weight: {optimal_weight}")

    start_time = time.time()

    # 1. Construct the optimal preference vector for all 6.6M nodes
    # We use a rapid list comprehension for memory efficiency
    years = g.vs['year']
    weights = [
        optimal_weight if str(y).isdigit() and int(y) >= target_year else 1.0
        for y in years
    ]

    # 2. Execute Personalized PageRank
    print("Computing Biased Teleportation Vector via PRPACK...")
    pr_fair_full = g.personalized_pagerank(directed=True, damping=0.85, reset=weights)

    print(f"Execution Completed in {time.time() - start_time:.2f} seconds.")

    # 3. Extract Top 10 Results
    top_10_indices = np.argsort(pr_fair_full)[::-1][:10]

    print("\n--- TOP 10 RECOMMENDED PAPERS (FAIR RANKING) ---")
    for rank, idx in enumerate(top_10_indices, 1):
        title = g.vs[idx]['title']
        year = g.vs[idx]['year']
        score = pr_fair_full[idx]
        print(f"{rank}. [{year}] {title[:75]}... (Score: {score:.4e})")

    print("\n[OBSERVATION]: As predicted by our Hub Gravity analysis, applying the weight of 1.2")
    print("at the macro-level failed to elevate modern papers. The top 10 remains entirely dominated")
    print("by historical hubs from the 1950s-1980s, proving the mathematical dilution effect.")

# Run the final scalable algorithm
run_fair_pagerank_full_graph(g, optimal_weight=1.2)

 FINAL EXECUTION: FAIRNESS-AWARE PAGERANK (FULL GRAPH)
Total Nodes: 6,611,132
Total Edges: 13,100,550
Applying Optimal Bias Weight: 1.2
Computing Biased Teleportation Vector via PRPACK...
Execution Completed in 13.56 seconds.

--- TOP 10 RECOMMENDED PAPERS (FAIR RANKING) ---
1. [1959] Finite Automata and Their Decision Problems... (Score: 1.1059e-03)
2. [1959] The Reduction of Two-Way Automata to One-Way Automata... (Score: 9.5666e-04)
3. [1965] Fuzzy Sets... (Score: 6.5538e-04)
4. [1976] New Directions in Cryptography... (Score: 6.0286e-04)
5. [1965] A Machine-Oriented Logic Based on the Resolution Principle... (Score: 4.8065e-04)
6. [1973] Recovery semantics for a DB/DC system.... (Score: 4.2566e-04)
7. [1973] Recovery scenario for a DB/DC system... (Score: 4.2383e-04)
8. [1983] Optimization by Simulated Annealing... (Score: 3.9749e-04)
9. [1992] A Training Algorithm for Optimal Margin Classifiers... (Score: 3.6152e-04)
10. [1967] On the Time Required to Perform Addition.... (Score: 

## 5. Final Conclusion

This project successfully exposed the mechanics of structural age bias in academic networks and tested the scalability of fairness-aware graph interventions.

**1. The Flaw (The Matthew Effect):** Using standard global centrality metrics (PageRank) and community detection (Louvain), we proved that traditional algorithms equate structural connectivity with importance. In directed acyclic citation networks, this creates an extreme bias: older papers hoard structural prestige simply by existing longer. As a result, over 70% of the graph—and the vast majority of modern innovation—is trapped in isolated structural silos.

**2. The Intervention (Fairness-Aware PageRank):** We implemented a demographic-aware teleportation vector to penalize historical entrenchment. On a 100,000-node core subgraph, this approach was highly successful. An empirical bias weight of 1.2 successfully bridged disconnected silos, restoring the visibility of modern research (raising representation from 6% to 42%) while preserving high semantic utility ($KT \approx 0.78$).

**3. The Scalability Limit (Structural Gravity):** Crucially, our final extreme-scale testing revealed the non-linear scaling of structural bias. Applying the same teleportation technique to the full 6.6-million node graph failed entirely. The historical "super-hubs" possessed such massive in-degree gravity that the teleportation bias was diluted and overpowered.

**Takeaway:** This project demonstrates that while modifying the global teleportation vector is an elegant, non-destructive way to inject fairness into local subgraphs, it is insufficient to overcome the deep historical gravity of macro-scale citation networks. True systemic fairness in billion-scale recommendation engines will likely require hybrid approaches that modify both global teleportation *and* local edge-transition probabilities simultaneously.

# GitHub Portfolio

Link to github: [github](https://github.com/BranHill21/branhill21_datamining_classproject)